# Notebook 01 — Residue Manifold Construction

Construct the mod30 residue manifold used by later notebooks. The goal is to make the ground-truth structure explicit before any learning or sampling experiment.

Core claim: after excluding 2, 3, and 5, prime residues occupy exactly eight mod30 lanes: `1, 7, 11, 13, 17, 19, 23, 29`.

## 1. Setup

This notebook uses SVG-only figure output. SVG is ideal here because these are structural line/point/bar figures that need to stay sharp in GitHub, docs, and papers.

In [ ]:

# Notebook 01 — Residue Manifold Construction

import os
from math import gcd

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Vector-friendly SVG defaults.
mpl.rcParams["svg.fonttype"] = "none"   # keep text editable/selectable in SVG
mpl.rcParams["figure.dpi"] = 120         # display only; SVG remains vector

# Canonical mod30 valid prime lanes: residues coprime to 30.
MOD = 30
N_MAX = 1000
VALID_LANES_MOD30 = [1, 7, 11, 13, 17, 19, 23, 29]

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)


def save_svg(fig, name):
    """Save a matplotlib figure as canonical SVG only."""
    path = f"figures/{name}.svg"
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")


## 2. Build residue and prime tables

The canonical valid lanes are the residue classes coprime to 30. The small primes 2, 3, and 5 are excluded from the prime count because they are the modulus factors, not recurring lanes.

In [ ]:

def simple_primes_upto(n):
    """Return primes <= n using a small sieve; no external dependencies."""
    if n < 2:
        return []
    sieve = np.ones(n + 1, dtype=bool)
    sieve[:2] = False
    for p in range(2, int(n**0.5) + 1):
        if sieve[p]:
            sieve[p*p:n+1:p] = False
    return np.flatnonzero(sieve).tolist()

primes = simple_primes_upto(N_MAX)
primes_excluding_2_3_5 = [p for p in primes if p not in (2, 3, 5)]

residue_df = pd.DataFrame({
    "n": np.arange(N_MAX + 1),
    "residue_mod30": np.arange(N_MAX + 1) % MOD,
})

prime_df = pd.DataFrame({
    "prime": primes_excluding_2_3_5,
    "residue_mod30": np.array(primes_excluding_2_3_5) % MOD,
})

lane_summary_df = pd.DataFrame({
    "residue_mod30": np.arange(MOD),
})
lane_summary_df["is_valid_prime_lane"] = lane_summary_df["residue_mod30"].isin(VALID_LANES_MOD30).astype(int)
lane_summary_df["gcd_with_30"] = lane_summary_df["residue_mod30"].apply(lambda r: gcd(int(r), MOD))
lane_summary_df["prime_count_up_to_N_excluding_2_3_5"] = (
    prime_df["residue_mod30"].value_counts().reindex(np.arange(MOD), fill_value=0).values
)

residue_df.to_csv("data/residues_mod30.csv", index=False)
prime_df.to_csv("data/primes_mod30.csv", index=False)
lane_summary_df.to_csv("data/residue_lane_summary_mod30.csv", index=False)

print("Valid lanes:", VALID_LANES_MOD30)
print("Lane density:", f"{len(VALID_LANES_MOD30)}/{MOD} = {len(VALID_LANES_MOD30)/MOD:.6f}")
print("Saved CSV files in data/")
lane_summary_df.head(30)


## 3. Ground-truth lane indicator

This binary figure is the definition figure: yellow cells mark valid mod30 prime lanes.

In [ ]:

# Ground-truth lane indicator: 1 for valid mod30 prime lanes, 0 otherwise.
indicator = lane_summary_df["is_valid_prime_lane"].to_numpy()[None, :]

fig, ax = plt.subplots(figsize=(12, 1.6))
ax.imshow(indicator, aspect="auto", interpolation="nearest")
ax.set_title("Ground-truth mod30 valid prime lane indicator")
ax.set_xlabel("Residue class r mod 30")
ax.set_yticks([0])
ax.set_yticklabels(["valid lane"])
ax.set_xticks(np.arange(MOD))

save_svg(fig, "residue_lane_matrix_mod30")
plt.show()


## 4. Circular residue manifold

All residue classes are embedded on a circle. Valid lanes are highlighted and connected in cyclic angular order.

In [ ]:

# Circular residue manifold: all 30 residues on S^1, with valid lanes highlighted.
all_residues = np.arange(MOD)
all_angles = 2 * np.pi * all_residues / MOD

valid = np.array(VALID_LANES_MOD30)
valid_angles = 2 * np.pi * valid / MOD
order = np.argsort(valid_angles)
valid_sorted = valid[order]
angles_sorted = valid_angles[order]

x_all, y_all = np.cos(all_angles), np.sin(all_angles)
x_valid, y_valid = np.cos(angles_sorted), np.sin(angles_sorted)

# Close cyclic path without changing canonical order of lane list elsewhere.
x_path = np.r_[x_valid, x_valid[0]]
y_path = np.r_[y_valid, y_valid[0]]

fig, ax = plt.subplots(figsize=(7.5, 7.5))

# reference circle
theta = np.linspace(0, 2 * np.pi, 512)
ax.plot(np.cos(theta), np.sin(theta), linewidth=1.2, alpha=0.45)

# all residues + valid lanes + cyclic valid-lane path
ax.scatter(x_all, y_all, s=38, alpha=0.55, label="all residues")
ax.plot(x_path, y_path, linewidth=2.2, alpha=0.75, label="cyclic valid-lane path")
ax.scatter(x_valid, y_valid, s=130, label="valid prime lanes", zorder=5)

# labels for all residue classes
for r, x, y in zip(all_residues, x_all, y_all):
    ax.text(1.105 * x, 1.105 * y, str(r), ha="center", va="center", fontsize=8)

ax.set_title("Residue Manifold: valid prime lanes in Z/30Z")
ax.set_aspect("equal")
ax.axis("off")
ax.legend(loc="upper right", frameon=True)

save_svg(fig, "residue_circle_mod30")
plt.show()


## 5. Prime residue histogram

The histogram verifies that all recurring prime support lies on the eight valid lanes.

In [ ]:

# Prime counts by residue class: support only occurs on 8 valid lanes.
counts_full = lane_summary_df["prime_count_up_to_N_excluding_2_3_5"].to_numpy()

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(np.arange(MOD), counts_full)
ax.set_title("Prime Residue Lanes mod 30")
ax.set_xlabel("Residue class r mod 30")
ax.set_ylabel("Prime count up to N, excluding 2, 3, 5")
ax.set_xticks(np.arange(MOD))
ax.set_ylim(0, max(counts_full) * 1.08)

save_svg(fig, "residue_histogram_mod30")
plt.show()


## 6. Numeric summary for downstream notebooks

In [ ]:

# Compact numeric summary used by later notebooks.
summary = {
    "modulus": MOD,
    "n_max": N_MAX,
    "valid_lane_count": len(VALID_LANES_MOD30),
    "total_residue_count": MOD,
    "lane_density": len(VALID_LANES_MOD30) / MOD,
    "valid_lanes": VALID_LANES_MOD30,
    "prime_count_excluding_2_3_5": len(primes_excluding_2_3_5),
}

pd.Series(summary)


## 7. Optional output bundle

Uncomment the final two lines when running in Colab to trigger a direct browser download. No Google Drive setup is needed.

In [ ]:

# --- Optional: Download Notebook 01 outputs (uncomment last line to trigger) ---

import os
import zipfile

zip_name = "01_residue_space_outputs.zip"
folders_to_zip = ["data", "figures"]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders_to_zip:
        if os.path.exists(folder):
            for root, _, filenames in os.walk(folder):
                for filename in filenames:
                    path = os.path.join(root, filename)
                    z.write(path, arcname=path)

print(f"Prepared: {zip_name}")

# from google.colab import files
# files.download(zip_name)
